In [ ]:
Q. 1 — Choose the Better Platform 
Ans: 
Requirement	                           Choice	         Reason
Large raw mixed data	               Data lake	     Flexible storage for many data shapes
Controlled SQL-ready business data	   Data warehouse	 Structured analytical use
Flexible storage plus managed          Lakehouse	     Combines lake-style storage with warehouse-style table capabilities
analytical tables 
Raw website events	                   Data lake	     Raw events can be retained before final use is known


In [ ]:
Q. 2 — Place BritMart Data 
Ans: 
One possible answer:

MySQL order extract -> lake or lakehouse raw area
supplier CSV -> lake or lakehouse raw area
website JSON -> lake or lakehouse raw area
monthly finance table -> warehouse or governed lakehouse table
daily sales summary -> warehouse or governed lakehouse table

The important point is to separate raw flexible storage needs from trusted analytical outputs.

In [ ]:
Q. 3 — Compare the Three Architectures 
Ans:
Area	                       Data Lake	                    Data Warehouse	                Lakehouse
Raw data support	           Strong	                        Usually not the main purpose	Strong
Strong structured reporting	   Possible but needs more design	Strong	                        Strong
Flexible data shapes	       Strong	                        More controlled	                Strong
Table-style analytics	       Depends on tools and design	    Strong	                        Strong

In [ ]:
Q. 4 — MCQ: Data Warehouse
Which requirement is the strongest fit for a data warehouse? 
Ans:  D. Run controlled business reporting over trusted structured data

In [ ]:
Q. 5 — Schema-on-Read or Schema-on-Write? 
Ans:
A -> Schema-on-read
B -> Schema-on-write
C -> Schema-on-read
D -> Schema-on-write

In [ ]:
Q. 6 — Design Bronze and Silver Rules 
Ans: 
Bronze
Keep the received records close to the source form so the team can inspect or replay them later. 

Silver
Apply agreed cleaning rules, for example: 
order_id must be present
amount_gbp must be positive
city should be trimmed
duplicate order_id values should be handled 

Bronze preserves received data. Silver creates a more trusted reusable dataset.

In [ ]:
Q. 7 — Process Three Batches from Bronze to Silver
Use Pandas.

In [2]:
import pandas as pd
 
batch_1 = pd.DataFrame(
    [
        (101, "London", 45.50, "complete"),
        (102, "Manchester", -5.00, "complete"),
    ],
    columns=[
        "order_id",
        "city",
        "amount_gbp",
        "status",
    ],
)
 
batch_2 = pd.DataFrame(
    [
        (103, " Bristol ", 30.00, "complete"),
        (104, "Leeds", 20.00, "cancelled"),
    ],
    columns=[
        "order_id",
        "city",
        "amount_gbp",
        "status",
    ],
)
 
batch_3 = pd.DataFrame(
    [
        (101, "London", 45.50, "complete"),
        (105, "Manchester", 51.00, "complete"),
    ],
    columns=[
        "order_id",
        "city",
        "amount_gbp",
        "status",
    ],
)
 
bronze = pd.concat(
    [batch_1, batch_2, batch_3],
    ignore_index=True,
)
 
silver = bronze[
    bronze["amount_gbp"] > 0
].copy()
 
silver["city"] = (
    silver["city"].str.strip()
)
 
silver = silver.drop_duplicates(
    subset=["order_id"],
    keep="first",
)
 
print("Bronze rows:", len(bronze))
print("Silver rows:", len(silver))
print(silver)

Bronze rows: 6
Silver rows: 4
   order_id        city  amount_gbp     status
0       101      London        45.5   complete
2       103     Bristol        30.0   complete
3       104       Leeds        20.0  cancelled
5       105  Manchester        51.0   complete


In [ ]:
Q. 8 — MCQ: Silver
Which record belongs most naturally in Silver? 
Ans: C. A cleaned and deduplicated order row

In [ ]:
Q. 9 — Build a Gold Summary
Using the Silver DataFrame from Q. 7: 

In [3]:
completed = silver[
    silver["status"] == "complete"
]
 
gold = (
    completed
    .groupby(
        "city",
        as_index=False,
    )
    .agg(
        revenue_gbp=(
            "amount_gbp",
            "sum",
        )
    )
    .sort_values(
        "revenue_gbp",
        ascending=False,
    )
)
 
print(gold)

         city  revenue_gbp
2  Manchester         51.0
1      London         45.5
0     Bristol         30.0


In [ ]:
Q. 10 — Structured, Semi-Structured, or Unstructured?
Ans: 
MySQL orders table -> Structured
JSON website event -> Semi-structured
CSV supplier file -> Structured
Product photo -> Unstructured
PDF supplier contract -> Unstructured

In [ ]:
Q. 11 — Design a Storage-First Flow
BritMart wants to keep received data before processing it. 
    Ans: 
flowchart LR
    A["MySQL Extract"] --> S["Storage"]
    B["Supplier CSV"] --> S
    C["Website JSON"] --> S
    S --> P["Processing"]
    P --> T["Trusted Output"]

In [ ]:
Q. 12 — MCQ: Gold
Which output is the strongest Gold example?
Ans: A. Daily revenue by city for reporting

In [ ]:
Q. 13 — Compute or Storage?
Classify each item. 
Ans:
A. Files kept for later use -> Storage
B. CPU and memory used to run a transformation -> Compute
C. Stored analytical table data -> Storage
D. Processing resources used for a query -> Compute 

Compute performs work. Storage keeps data.

In [ ]:
Q. 14 — Why Separate Compute and Storage? 
Ans: The stored data can remain available even when large processing resources are not running.
When more processing power is needed, compute can be increased for the workload without redesigning where all data is stored.
This can improve flexibility and cost control.

In [ ]:
Q. 15 — Create a Small Metadata Catalogue 

Ans:
Dataset          	Purpose	Main columns	 Source	                          Owner/team	    Last update
bronze_orders	    Received order data	     order_id, city, amount, status	  Source batches	Data Engineering	2026-08-01
silver_orders	    Clean reusable orders	 order_id, city, amount, status	  bronze_orders	    Data Engineering	2026-08-01
gold_daily_sales	Reporting summary	     city, revenue_gbp	              silver_orders	    Analytics/Data	2026-08-01
A catalogue helps users find and understand datasets without opening every file or table.

In [ ]:
Q. 16 — MCQ: Metadata
Why is metadata useful? 
Ans: B. It describes data so people and systems can understand what the data represents.

In [ ]:
Q. 17 — Trace One Record Through Bronze, Silver, and Gold 
Ans: 
Bronze
9001 | " London " | 50 | complete 
Keep it close to the received form.

Silver
9001 | London | 50 | complete 
The city is cleaned and the row passes validation.

Gold
The row contributes £50 to London's reporting total.

Gold may not keep the original order row if the required output is an aggregate.

In [ ]:
Q. 18 — Find a Layering Mistake
A team creates Gold directly from raw Bronze data and repeats different cleaning rules in five reports.
Ans: 
Possible problems:

different reports may clean the same source differently;
duplicate or invalid data may enter business totals;
cleaning logic is repeated in many places;
fixing a rule requires changing several reports.
A reusable Silver layer gives the reports one trusted cleaned source.

In [ ]:
Q. 19 — Reconcile the Layers 
Ans:
Rejected/removed rows = 25 + 15 = 40
Expected Silver rows = 1000 - 40 = 960

In [ ]:
Q. 20 — Design a Small Platform
Create one architecture using only today's concepts. 
Ans:
One possible design:
flowchart LR
    A["MySQL"] --> B["Bronze"]
    C["CSV"] --> B
    D["JSON"] --> B
    B --> E["Silver"]
    E --> F["Gold"]
    F --> G["Reporting"]
    S["Storage"] --- B
    S --- E
    S --- F
    C1["Compute"] --- E
    C1 --- F
    M["Metadata / Catalogue"] --- B
    M --- E
    M --- F

Bronze: keeps received data.
Silver: keeps cleaned reusable data.
Gold: keeps business-ready output.
Storage: keeps the data.
Compute: performs processing.
Metadata/catalogue: describes datasets and helps users find them.
Reporting: uses trusted Gold data.